In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
!pip install lightgbm -q

In [3]:
import pandas as pd
import numpy as np
import joblib
import os

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import lightgbm as lgb

In [5]:
# Paths
BASE_PATH = "/content/drive/MyDrive/Zero-Day-IoT-Project"

DATA_PATH = BASE_PATH + "/01_Data"
MODEL_PATH = BASE_PATH + "/03_Models"
REPORT_PATH = BASE_PATH + "/04_Reports"

os.makedirs(MODEL_PATH, exist_ok=True)
os.makedirs(REPORT_PATH, exist_ok=True)

In [6]:
# Load Dataset
df = pd.read_parquet(DATA_PATH + "/known_train.parquet")

print("Dataset Shape:", df.shape)
df.head()

Dataset Shape: (121293, 53)


,arp.opcode,arp.hw.size,icmp.checksum,icmp.seq_le,icmp.unused,http.content_length,http.request.method,http.referer,http.request.version,http.response,...,mqtt.proto_len,mqtt.protoname,mqtt.topic,mqtt.topic_len,mqtt.ver,mbtcp.len,mbtcp.trans_id,mbtcp.unit_id,Attack_label,Attack_type
2215,0.0,0.0,0.0,0.0,0.0,0.0,1,2,3,0.0,...,0.0,1,1,0.0,0.0,0.0,0.0,0.0,1,10
2216,0.0,0.0,0.0,0.0,0.0,0.0,1,2,3,0.0,...,0.0,1,1,0.0,0.0,0.0,0.0,0.0,1,10
2217,0.0,0.0,0.0,0.0,0.0,0.0,1,2,3,0.0,...,0.0,1,1,0.0,0.0,0.0,0.0,0.0,1,10
2218,0.0,0.0,0.0,0.0,0.0,0.0,1,2,3,0.0,...,0.0,1,1,0.0,0.0,0.0,0.0,0.0,1,10
2219,0.0,0.0,0.0,0.0,0.0,0.0,1,2,3,0.0,...,0.0,1,1,0.0,0.0,0.0,0.0,0.0,1,10


In [7]:
# Prepare Features & Target
X = df.drop(columns=["Attack_type", "Attack_label"])
y = df["Attack_type"]

print("X Shape:", X.shape)
print("Classes:", y.nunique())

X Shape: (121293, 51)
Classes: 10


In [8]:
# Train Test Split
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print(X_train.shape, X_test.shape)

(97034, 51) (24259, 51)


In [9]:
# Train LightGBM
model = lgb.LGBMClassifier(
    objective="multiclass",
    num_class=y.nunique(),
    n_estimators=300,
    learning_rate=0.05,
    max_depth=10,
    num_leaves=50,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42
)

model.fit(X_train, y_train)

print("Model Trained Successfully")

[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.041201 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 3408
[LightGBM] [Info] Number of data points in the train set: 97034, number of used features: 40
[LightGBM] [Info] Start training from score -2.509085
[LightGBM] [Info] Start training from score -2.441250
[LightGBM] [Info] Start training from score -2.225879
[LightGBM] [Info] Start training from score -2.471171
[LightGBM] [Info] Start training from score -2.124229
[LightGBM] [Info] Start training from score -1.613869
[LightGBM] [Info] Start training from score -2.497872
[LightGBM] [Info] Start training from score -2.609769
[LightGBM] [Info] Start training from score -2.527111
[LightGBM] [Info] Start training from score -2.467397
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further 

In [10]:
# Predictions
y_pred = model.predict(X_test)

acc = accuracy_score(y_test, y_pred)

print("Accuracy:", round(acc*100, 2), "%")

Accuracy: 99.99 %


In [11]:
# Classification Report
report = classification_report(y_test, y_pred, output_dict=True)

report_df = pd.DataFrame(report).transpose()

report_df

,precision,recall,f1-score,support
0,1.000000,0.999493,0.999747,1973.000000
1,1.000000,1.000000,1.000000,2112.000000
2,1.000000,0.999618,0.999809,2619.000000
3,1.000000,1.000000,1.000000,2049.000000
4,1.000000,1.000000,1.000000,2900.000000
7,1.000000,1.000000,1.000000,4831.000000
8,0.999499,1.000000,0.999750,1996.000000
9,0.999440,1.000000,0.999720,1784.000000
10,0.999484,1.000000,0.999742,1938.000000
11,1.000000,0.999514,0.999757,2057.000000


In [12]:
# Save Metrics
report_df.to_csv(REPORT_PATH + "/lightgbm_metrics.csv", index=True)

print("Metrics Saved")

Metrics Saved


In [13]:
# Confusion Matrix
cm = confusion_matrix(y_test, y_pred)

print(cm)

[[1972    0    0    0    0    0    0    0    1    0]
 [   0 2112    0    0    0    0    0    0    0    0]
 [   0    0 2618    0    0    0    0    1    0    0]
 [   0    0    0 2049    0    0    0    0    0    0]
 [   0    0    0    0 2900    0    0    0    0    0]
 [   0    0    0    0    0 4831    0    0    0    0]
 [   0    0    0    0    0    0 1996    0    0    0]
 [   0    0    0    0    0    0    0 1784    0    0]
 [   0    0    0    0    0    0    0    0 1938    0]
 [   0    0    0    0    0    0    1    0    0 2056]]


In [14]:
# Save Model
joblib.dump(model, MODEL_PATH + "/lightgbm_known.pkl")

print("Model Saved Successfully")

Model Saved Successfully


In [15]:
# Feature Importance
importance = pd.DataFrame({
    "Feature": X.columns,
    "Importance": model.feature_importances_
}).sort_values(by="Importance", ascending=False)

importance.head(20)

,Feature,Importance
25,tcp.srcport,13467
22,tcp.options,12777
18,tcp.dstport,8849
11,tcp.ack,7849
12,tcp.ack_raw,6417
13,tcp.checksum,6109
24,tcp.seq,4226
6,http.request.method,2213
21,tcp.len,2181
27,udp.stream,2009


In [16]:
# Save Feature Importance
importance.to_csv(REPORT_PATH + "/feature_importance.csv", index=False)

print("Feature Importance Saved")

Feature Importance Saved


In [17]:
print("FILES CREATED")
print("----------------")
print("03_Models/lightgbm_known.pkl")
print("04_Reports/lightgbm_metrics.csv")
print("04_Reports/feature_importance.csv")

FILES CREATED
----------------
03_Models/lightgbm_known.pkl
04_Reports/lightgbm_metrics.csv
04_Reports/feature_importance.csv
